# Phase 0.1 — Data Sampling & Cleaning

This notebook performs the data filtering pipeline from the execution plan:
1. Filter to `type = "game"` only
2. Filter users with >= 3 reviews
3. Filter games with >= 2 reviews from qualifying users
4. Canonicalise genres/categories to English
5. Clean data quality issues (`votes_funny` overflow, price outliers)
6. Apply transformations (log playtime, log price)
7. Drop zero-variance columns
8. Save final cleaned dataset

## 0. Setup

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import pandas as pd
import numpy as np
import os

DATA_DIR = "/content/drive/MyDrive/steam_dataset_2025_csv"
OUTPUT_DIR = "/content/drive/MyDrive/processed_steam_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load tables
apps = pd.read_csv(os.path.join(DATA_DIR, "applications.csv"), low_memory=False)
reviews = pd.read_csv(os.path.join(DATA_DIR, "reviews.csv"))
genres = pd.read_csv(os.path.join(DATA_DIR, "genres.csv"))
categories = pd.read_csv(os.path.join(DATA_DIR, "categories.csv"))
app_genres = pd.read_csv(os.path.join(DATA_DIR, "application_genres.csv"))
app_categories = pd.read_csv(os.path.join(DATA_DIR, "application_categories.csv"))
app_developers = pd.read_csv(os.path.join(DATA_DIR, "application_developers.csv"))
app_publishers = pd.read_csv(os.path.join(DATA_DIR, "application_publishers.csv"))
app_platforms = pd.read_csv(os.path.join(DATA_DIR, "application_platforms.csv"))
platforms = pd.read_csv(os.path.join(DATA_DIR, "platforms.csv"))
developers = pd.read_csv(os.path.join(DATA_DIR, "developers.csv"))
publishers = pd.read_csv(os.path.join(DATA_DIR, "publishers.csv"))

print(f"Raw data: {len(apps):,} apps, {len(reviews):,} reviews")

Raw data: 239,664 apps, 1,048,148 reviews


## 1. Three-Step Interaction Filtering

In [20]:
# Step 1: Keep only type='game'
game_appids = set(apps.loc[apps["type"] == "game", "appid"])
reviews_f = reviews[reviews["appid"].isin(game_appids)].copy()
print(f"Step 1 — type='game':  {len(reviews_f):,} reviews | {reviews_f['appid'].nunique():,} games | {reviews_f['author_steamid'].nunique():,} users")

# Step 2: Keep users with >= 3 reviews
user_counts = reviews_f.groupby("author_steamid").size()
qualifying_users = user_counts[user_counts >= 3].index
reviews_f = reviews_f[reviews_f["author_steamid"].isin(qualifying_users)]
print(f"Step 2 — users >= 3:   {len(reviews_f):,} reviews | {reviews_f['appid'].nunique():,} games | {reviews_f['author_steamid'].nunique():,} users")

# Step 3: Keep games with >= 2 reviews from qualifying users
game_counts = reviews_f.groupby("appid").size()
qualifying_games = game_counts[game_counts >= 2].index
reviews_f = reviews_f[reviews_f["appid"].isin(qualifying_games)]
print(f"Step 3 — games >= 2:   {len(reviews_f):,} reviews | {reviews_f['appid'].nunique():,} games | {reviews_f['author_steamid'].nunique():,} users")

# Sparsity
n_users = reviews_f["author_steamid"].nunique()
n_games = reviews_f["appid"].nunique()
n_interactions = len(reviews_f)
sparsity = 1 - (n_interactions / (n_users * n_games))
print(f"\nFinal: {n_interactions:,} interactions, sparsity = {sparsity*100:.4f}%")

Step 1 — type='game':  865,782 reviews | 85,529 games | 600,542 users
Step 2 — users >= 3:   250,379 reviews | 58,367 games | 36,996 users
Step 3 — games >= 2:   224,895 reviews | 32,883 games | 36,840 users

Final: 224,895 interactions, sparsity = 99.9814%


**Caveat**:

Since steps 2 and 3 run sequentially, not iteratively. Step 3 can remove games, which may drop some users below the >= 3 review threshold enforced in step 2. For example, if user X had exactly 3 reviews (games A, B, C) after step 2, but game C is removed in step 3 (only 1 review from qualifying users), user X now has 2 reviews — violating the original constraint. The output confirms this: `min=1` in reviews per user. To strictly enforce both constraints, steps 2 and 3 would need to loop until no more rows are removed.

## 2. Filter Junction Tables to Qualifying Games

In [21]:
final_game_ids = set(reviews_f["appid"].unique())
final_user_ids = set(reviews_f["author_steamid"].unique())

# Filter apps table
apps_f = apps[apps["appid"].isin(final_game_ids)].copy()

# Filter junction tables to qualifying games
app_genres_f = app_genres[app_genres["appid"].isin(final_game_ids)].copy()
app_categories_f = app_categories[app_categories["appid"].isin(final_game_ids)].copy()
app_developers_f = app_developers[app_developers["appid"].isin(final_game_ids)].copy()
app_publishers_f = app_publishers[app_publishers["appid"].isin(final_game_ids)].copy()
app_platforms_f = app_platforms[app_platforms["appid"].isin(final_game_ids)].copy()

print(f"Filtered apps:           {len(apps_f):,}")
print(f"Filtered app_genres:     {len(app_genres_f):,}")
print(f"Filtered app_categories: {len(app_categories_f):,}")
print(f"Filtered app_developers: {len(app_developers_f):,}")
print(f"Filtered app_publishers: {len(app_publishers_f):,}")
print(f"Filtered app_platforms:  {len(app_platforms_f):,}")

Filtered apps:           32,883
Filtered app_genres:     92,483
Filtered app_categories: 157,011
Filtered app_developers: 35,771
Filtered app_publishers: 34,385
Filtered app_platforms:  44,960


## 3. Canonicalise Genres & Categories to English

The genre/category lookup tables contain multilingual duplicates — the same logical
genre (e.g. "Strategy") appears as separate rows in multiple languages, each with its
own unique ID:

| id | name |
|----|------|
| 5 | Strategy |
| 4 | Strategie (German) |
| 30 | Stratégie (French) |
| 56 | ストラテジー (Japanese) |
| 84 | 策略 (Chinese) |

Since there is no shared key linking translations of the same concept, we apply a
**two-pass filter**:

1. **ASCII filter**: Remove names with non-ASCII characters (catches CJK, Cyrillic,
   accented Latin). This is data-driven and robust to new entries.
2. **Minimum-count threshold**: Some non-English names are ASCII (e.g. "Strategie",
   "Azione", "Aventura"). These affect only 1–3 games each, while all English genres
   have 23+ games and English categories have 8+ games. A threshold of 6 cleanly
   separates them without requiring manual curation.
3. **Exception safelist**: A small set of legitimate English names that fall below the
   threshold — "Accounting" (3 games), "Movie" (1 game), "Mods" (2 games), and
   "Mods (require HL2)" (1 game). These are stable Steam platform terms unlikely to
   change, and are far fewer to maintain than a full allowlist of 90+ entries.

In [22]:
# === Pass 1: ASCII filter ===
# Non-English translations using non-ASCII characters (CJK, Cyrillic, accented Latin)
# are removed. Some ASCII non-English names (e.g. "Strategie") pass through.
def is_ascii(text: str) -> bool:
    return all(ord(c) < 128 for c in str(text))

english_genre_ids = set(genres[genres["name"].apply(is_ascii)]["id"])
english_cat_ids = set(categories[categories["name"].apply(is_ascii)]["id"])

app_genres_f = app_genres_f[app_genres_f["genre_id"].isin(english_genre_ids)].copy()
app_categories_f = app_categories_f[app_categories_f["category_id"].isin(english_cat_ids)].copy()

# Add readable names
genre_id_to_name = genres.set_index("id")["name"].to_dict()
cat_id_to_name = categories.set_index("id")["name"].to_dict()

app_genres_f["genre_name"] = app_genres_f["genre_id"].map(genre_id_to_name)
app_categories_f["category_name"] = app_categories_f["category_id"].map(cat_id_to_name)

print(f"After ASCII filter:")
print(f"  Genres: {app_genres_f['genre_name'].nunique()}, Categories: {app_categories_f['category_name'].nunique()}")

# === Pass 2: Minimum-count threshold ===
# English genres have 23+ games, non-English ASCII names have 1-3.
# English categories have 8+ games, non-English ASCII names have 1-5.
# A threshold of 6 cleanly separates them.
# A small safelist covers legitimate English names that fall below the threshold.
MIN_COUNT = 6
ENGLISH_EXCEPTIONS = {"Accounting", "Movie", "Mods", "Mods (require HL2)"}

genre_counts = app_genres_f.groupby("genre_name")["appid"].nunique().sort_values(ascending=False)
valid_genres = set(genre_counts[genre_counts >= MIN_COUNT].index) | (ENGLISH_EXCEPTIONS & set(genre_counts.index))
dropped_genres = genre_counts[~genre_counts.index.isin(valid_genres)]

cat_counts = app_categories_f.groupby("category_name")["appid"].nunique().sort_values(ascending=False)
valid_cats = set(cat_counts[cat_counts >= MIN_COUNT].index) | (ENGLISH_EXCEPTIONS & set(cat_counts.index))
dropped_cats = cat_counts[~cat_counts.index.isin(valid_cats)]

app_genres_f = app_genres_f[app_genres_f["genre_name"].isin(valid_genres)].copy()
app_categories_f = app_categories_f[app_categories_f["category_name"].isin(valid_cats)].copy()

print(f"\nAfter minimum-count filter (>= {MIN_COUNT} games, + {len(ENGLISH_EXCEPTIONS)} English exceptions):")
print(f"  Genres: {app_genres_f['genre_name'].nunique()}, Categories: {app_categories_f['category_name'].nunique()}")

print(f"\nDropped {len(dropped_genres)} genres:")
for name, count in dropped_genres.items():
    print(f"  {name:<30} {count} games")

print(f"\nDropped {len(dropped_cats)} categories:")
for name, count in dropped_cats.items():
    print(f"  {name:<55} {count} games")

print(f"\nGenre assignments: {len(app_genres_f):,}")
print(f"Category assignments: {len(app_categories_f):,}")

print(f"\nTop 10 genres:")
print(app_genres_f["genre_name"].value_counts().head(10).to_string())

After ASCII filter:
  Genres: 44, Categories: 120

After minimum-count filter (>= 6 games, + 4 English exceptions):
  Genres: 28, Categories: 58

Dropped 16 genres:
  Aventura                       3 games
  Acesso Antecipado              2 games
  Aventure                       2 games
  Akcja                          1 games
  Azione                         1 games
  Corrida                        1 games
  Estrategia                     1 games
  Eventyr                        1 games
  Przygodowe                     1 games
  Petualangan                    1 games
  Rekreacyjne                    1 games
  Rollespill                     1 games
  Sportowe                       1 games
  Symulacje                      1 games
  Strategia                      1 games
  Strategie                      1 games

Dropped 62 categories:
  Um jogador                                              5 games
  Nuvem Steam                                             4 games
  Compat. total com con

## 4. Data Quality Cleaning

In [23]:
# 4a. Clean votes_funny overflow (max 4.29B suggests integer overflow)
reviews_f["votes_funny"] = pd.to_numeric(reviews_f["votes_funny"], errors="coerce")
cap_funny = reviews_f["votes_funny"].quantile(0.999)
n_capped_funny = (reviews_f["votes_funny"] > cap_funny).sum()
reviews_f["votes_funny"] = reviews_f["votes_funny"].clip(upper=cap_funny)
print(f"votes_funny: capped {n_capped_funny:,} values at {cap_funny:.0f}")

# 4b. Clean mat_final_price outliers (cap at 99th percentile)
price_cap = apps_f["mat_final_price"].quantile(0.99)
n_capped_price = (apps_f["mat_final_price"] > price_cap).sum()
apps_f["mat_final_price"] = apps_f["mat_final_price"].clip(upper=price_cap)
apps_f["mat_initial_price"] = apps_f["mat_initial_price"].clip(upper=price_cap)
print(f"mat_final_price: capped {n_capped_price:,} values at {price_cap:.0f} cents (${price_cap/100:.2f})")

# 4c. Drop steam_purchase (100% True = zero variance)
reviews_f = reviews_f.drop(columns=["steam_purchase"])
print(f"Dropped steam_purchase (zero variance)")

# 4d. Convert voted_up to boolean if stored as string
reviews_f["voted_up"] = reviews_f["voted_up"].map({True: True, False: False, "True": True, "False": False})
print(f"\nvoted_up distribution: {reviews_f['voted_up'].value_counts().to_dict()}")

votes_funny: capped 219 values at 42
mat_final_price: capped 311 values at 7181 cents ($71.81)
Dropped steam_purchase (zero variance)

voted_up distribution: {True: 168712, False: 56183}


## 5. Log Transformations

Apply `log(1 + x)` to heavily right-skewed numeric features.

In [24]:
# Playtime columns in reviews (log-transform)
playtime_cols = ["author_playtime_forever", "author_playtime_last_two_weeks", "author_playtime_at_review"]
for col in playtime_cols:
    reviews_f[f"{col}_log"] = np.log1p(reviews_f[col])
    print(f"{col}_log: min={reviews_f[f'{col}_log'].min():.2f}, median={reviews_f[f'{col}_log'].median():.2f}, max={reviews_f[f'{col}_log'].max():.2f}")

# Price in apps (log-transform)
apps_f["mat_final_price_log"] = np.log1p(apps_f["mat_final_price"])
print(f"\nmat_final_price_log: min={apps_f['mat_final_price_log'].min():.2f}, median={apps_f['mat_final_price_log'].median():.2f}, max={apps_f['mat_final_price_log'].max():.2f}")

author_playtime_forever_log: min=0.00, median=5.08, max=15.28
author_playtime_last_two_weeks_log: min=0.00, median=0.00, max=9.91
author_playtime_at_review_log: min=0.69, median=4.80, max=13.41

mat_final_price_log: min=3.91, median=6.40, max=8.88


## 6. Summary of Final Dataset

In [25]:
print("=" * 60)
print("FINAL FILTERED DATASET SUMMARY")
print("=" * 60)

n_users = reviews_f["author_steamid"].nunique()
n_games = reviews_f["appid"].nunique()
n_interactions = len(reviews_f)
sparsity = 1 - (n_interactions / (n_users * n_games))

print(f"\n--- Interactions ---")
print(f"Users:          {n_users:,}")
print(f"Games:          {n_games:,}")
print(f"Interactions:   {n_interactions:,}")
print(f"Sparsity:       {sparsity*100:.4f}%")
print(f"Avg reviews/user: {n_interactions/n_users:.1f}")
print(f"Avg reviews/game: {n_interactions/n_games:.1f}")

print(f"\n--- Label Distribution ---")
vc = reviews_f["voted_up"].value_counts()
print(f"Positive (voted_up=True):  {vc.get(True, 0):,} ({vc.get(True, 0)/n_interactions*100:.1f}%)")
print(f"Negative (voted_up=False): {vc.get(False, 0):,} ({vc.get(False, 0)/n_interactions*100:.1f}%)")

print(f"\n--- Reviews Table ---")
print(f"Shape: {reviews_f.shape}")
print(f"Columns: {list(reviews_f.columns)}")

print(f"\n--- Apps Table ---")
print(f"Shape: {apps_f.shape}")

print(f"\n--- Junction Tables ---")
print(f"Genre assignments:    {len(app_genres_f):,} (English only)")
print(f"Category assignments: {len(app_categories_f):,} (English only)")
print(f"Developer links:      {len(app_developers_f):,}")
print(f"Publisher links:      {len(app_publishers_f):,}")
print(f"Platform links:       {len(app_platforms_f):,}")

FINAL FILTERED DATASET SUMMARY

--- Interactions ---
Users:          36,840
Games:          32,883
Interactions:   224,895
Sparsity:       99.9814%
Avg reviews/user: 6.1
Avg reviews/game: 6.8

--- Label Distribution ---
Positive (voted_up=True):  168,712 (75.0%)
Negative (voted_up=False): 56,183 (25.0%)

--- Reviews Table ---
Shape: (224895, 25)
Columns: ['recommendationid', 'appid', 'author_steamid', 'author_num_games_owned', 'author_num_reviews', 'author_playtime_forever', 'author_playtime_last_two_weeks', 'author_playtime_at_review', 'author_last_played', 'language', 'review_text', 'timestamp_created', 'timestamp_updated', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'received_for_free', 'written_during_early_access', 'created_at', 'updated_at', 'author_playtime_forever_log', 'author_playtime_last_two_weeks_log', 'author_playtime_at_review_log']

--- Apps Table ---
Shape: (32883, 31)

--- Junction Tables ---
Genre assignments:    92,438 (English onl

In [26]:
# Distribution of reviews per user and per game (after filtering)
reviews_per_user = reviews_f.groupby("author_steamid").size()
reviews_per_game = reviews_f.groupby("appid").size()

print("Reviews per USER:")
print(reviews_per_user.describe().to_string())
print(f"\nReviews per GAME:")
print(reviews_per_game.describe().to_string())

Reviews per USER:
count    36840.000000
mean         6.104642
std         14.477609
min          1.000000
25%          3.000000
50%          4.000000
75%          5.000000
max        972.000000

Reviews per GAME:
count    32883.000000
mean         6.839248
std          7.642541
min          2.000000
25%          2.000000
50%          4.000000
75%          8.000000
max        100.000000


In [27]:
# Show sample rows
print("Sample reviews (first 5):")
display(reviews_f.head())

print("\nSample apps (first 5):")
display(apps_f[["appid", "name", "is_free", "short_description", "metacritic_score",
                "mat_final_price", "mat_final_price_log"]].head())

Sample reviews (first 5):


,recommendationid,appid,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played,language,...,votes_funny,weighted_vote_score,comment_count,received_for_free,written_during_early_access,created_at,updated_at,author_playtime_forever_log,author_playtime_last_two_weeks_log,author_playtime_at_review_log
0,10000000,264220,76561198085405844,760,74,12.0,0.0,12.0,1.399060e+09,polish,...,1,0.459906,0,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00,2.564949,0.0,2.564949
1,100001066,1006440,76561198014439859,485,234,424.0,0.0,424.0,1.632666e+09,russian,...,0,0.630770,0,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00,6.052089,0.0,6.052089
4,100002504,1338560,76561198138996331,816,26,25.0,0.0,25.0,1.632639e+09,english,...,0,0.523810,0,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00,3.258097,0.0,3.258097
12,100005649,1557350,76561198881245121,0,32,1299.0,0.0,500.0,1.655569e+09,english,...,0,0.571614,0,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00,7.170120,0.0,6.216606
17,100008287,1155960,76561198034565793,3027,401,52.0,0.0,52.0,1.578691e+09,russian,...,0,0.512195,0,False,False,2025-09-07 12:51:00.564782+00:00,2025-09-08 00:47:55.754043+00:00,3.970292,0.0,3.970292



Sample apps (first 5):


,appid,name,is_free,short_description,metacritic_score,mat_final_price,mat_final_price_log
18,400,Portal,False,Portal™ is a new single player game from Valve...,90.0,999.0,6.907755
35,1300,SiN Episodes: Emergence,False,"You are John Blade, commander of HardCorps, an...",75.0,999.0,6.907755
36,1313,SiN: Gold,False,SiN: Gold has returned! Free update for origin...,NaN,999.0,6.907755
40,1520,DEFCON,False,"Inspired by the 1983 cult classic film, Wargam...",84.0,1199.0,7.090077
48,1640,Disciples II: Gallean's Return,False,Disciples II: Gallean's Return is a compilatio...,84.0,599.0,6.396930


## 7. Save Processed Data

In [28]:
# Save all filtered tables as parquet for fast loading in subsequent phases
reviews_f.to_parquet(os.path.join(OUTPUT_DIR, "reviews_filtered.parquet"), index=False)
apps_f.to_parquet(os.path.join(OUTPUT_DIR, "apps_filtered.parquet"), index=False)
app_genres_f.to_parquet(os.path.join(OUTPUT_DIR, "app_genres_filtered.parquet"), index=False)
app_categories_f.to_parquet(os.path.join(OUTPUT_DIR, "app_categories_filtered.parquet"), index=False)
app_developers_f.to_parquet(os.path.join(OUTPUT_DIR, "app_developers_filtered.parquet"), index=False)
app_publishers_f.to_parquet(os.path.join(OUTPUT_DIR, "app_publishers_filtered.parquet"), index=False)
app_platforms_f.to_parquet(os.path.join(OUTPUT_DIR, "app_platforms_filtered.parquet"), index=False)

# Also save the lookup tables (genres, categories, developers, publishers with names)
genres.to_parquet(os.path.join(OUTPUT_DIR, "genres.parquet"), index=False)
categories.to_parquet(os.path.join(OUTPUT_DIR, "categories.parquet"), index=False)
developers.to_parquet(os.path.join(OUTPUT_DIR, "developers.parquet"), index=False)
publishers.to_parquet(os.path.join(OUTPUT_DIR, "publishers.parquet"), index=False)
platforms.to_parquet(os.path.join(OUTPUT_DIR, "platforms.parquet"), index=False)

print(f"All tables saved to {OUTPUT_DIR}/")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"  {f:<40} {size_mb:.1f} MB")

All tables saved to /content/drive/MyDrive/processed_steam_data/
  app_categories_filtered.parquet          0.6 MB
  app_developers_filtered.parquet          0.4 MB
  app_genres_filtered.parquet              0.4 MB
  app_platforms_filtered.parquet           0.2 MB
  app_publishers_filtered.parquet          0.4 MB
  apps_filtered.parquet                    8.6 MB
  categories.parquet                       0.0 MB
  developers.parquet                       2.1 MB
  genres.parquet                           0.0 MB
  platforms.parquet                        0.0 MB
  publishers.parquet                       1.8 MB
  reviews_filtered.parquet                 104.7 MB
